# Lab 01 — Steganography

This notebook walks through steganography step by step.

**Pipeline:**
image → inspect pixels → encode message (LSB) → detect difference → decode message

Goal: understand how tiny changes to pixel values can hide and carry data that is invisible to the human eye.

---
### How to work in this lab
For each section:
1. Read the short explanation.
2. Run the code cell.
3. Observe the output.
4. Try changing a parameter and re-run.
5. Write one sentence: **what changed and why?**


## Cell 1 — Imports and display function

### What this cell does
- Imports OpenCV, NumPy, Matplotlib
- Defines a helper `show()` function to display images correctly

### Why it matters
We need to see every pixel change. A stable display function means we can inspect results after every step.

### What to do
- Run the cell, do not modify it.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(imgs, titles=None, cols=None):
    if not isinstance(imgs, (list, tuple)):
        imgs = [imgs]
        titles = [titles or ""]
    if titles is None:
        titles = [""] * len(imgs)
    n = len(imgs)
    cols = cols or n
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).flatten()
    for ax, img, title in zip(axes, imgs, titles):
        if len(img.shape) == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("Imports ready.")


## Cell 2 — Create the base image

### What this cell does
- Creates a simple synthetic image with geometric shapes
- This is our "carrier image" — the image we will hide data inside

### Why it matters
Steganography requires a carrier image. We use a synthetic one so the lab works for everyone without an upload.

### What to observe
- The pixel values range from 0 (black) to 255 (white)
- Simple shapes mean we can reason easily about what changed


In [ ]:
# Create a carrier image — 200x300 with simple shapes
np.random.seed(42)
img = np.ones((200, 300, 3), dtype=np.uint8) * 180  # light gray background

# Add some structure
cv2.rectangle(img, (20, 20), (130, 100), (60, 90, 200), -1)    # blue rectangle
cv2.circle(img, (220, 100), 60, (50, 180, 80), -1)             # green circle
cv2.rectangle(img, (20, 120), (280, 180), (200, 80, 50), -1)   # red bar

show(img, "Carrier image")
print(f"Image shape: {img.shape}  — height={img.shape[0]}, width={img.shape[1]}, channels={img.shape[2]}")
print(f"Total pixels: {img.shape[0] * img.shape[1]:,}")


## Cell 3 — Pixel binary inspection

### Concept
Every pixel value is a number from 0 to 255. In memory it is stored as 8 bits (one byte).

**Example:** value 120 → binary `01111000`

The **Least Significant Bit (LSB)** is the rightmost bit. Flipping it changes the value by only ±1 — completely invisible to the human eye.

### What this cell does
- Reads 5 pixel values from the image
- Prints each value in decimal AND in 8-bit binary
- Highlights the LSB of each channel

### Why it matters
Understanding the bit structure is the foundation of LSB steganography. The LSB is the hiding place.


In [ ]:
# Sample 5 pixels and display their binary representation
sample_pixels = [(50, 50), (100, 150), (180, 250), (20, 220), (90, 30)]

print(f"{'Position':<12} {'Channel':<8} {'Dec':>5}  {'Binary':>10}  LSB")
print("-" * 50)

for (row, col) in sample_pixels:
    b, g, r = img[row, col]
    for name, val in zip(['Blue', 'Green', 'Red'], [b, g, r]):
        binary = format(int(val), '08b')
        lsb = binary[-1]
        print(f"({row:3},{col:3})    {name:<8} {int(val):>5}   {binary}    {lsb}")
    print()


## Cell 4 — Storage capacity

### Concept
If we hide 1 bit per channel per pixel, and an image has W×H pixels with 3 channels:
- **Total bits available** = W × H × 3
- **Total characters** = total bits ÷ 8 (each ASCII character = 8 bits)

### What this cell does
- Calculates the maximum message length for our image
- Compares single-channel vs multi-channel capacity

### Why it matters
Before encoding, an engineer must verify the carrier has enough capacity for the message.


In [ ]:
h, w, c = img.shape
total_pixels = h * w

# Single channel (Red only)
bits_single = total_pixels * 1
chars_single = bits_single // 8

# All 3 channels
bits_multi = total_pixels * 3
chars_multi = bits_multi // 8

print(f"Image: {w}x{h} = {total_pixels:,} pixels")
print()
print(f"Single channel (Red only):")
print(f"  Available bits:       {bits_single:,}")
print(f"  Max message length:   {chars_single:,} characters")
print()
print(f"All three channels (B, G, R):")
print(f"  Available bits:       {bits_multi:,}")
print(f"  Max message length:   {chars_multi:,} characters")
print()
print(f"Multi-channel stores {c}x more data per pixel.")


## Cell 5 — LSB Encoding (single channel)

### Concept
To hide a character 'H' (ASCII 72 = `01001000`):
1. Take 8 consecutive pixels
2. For each pixel's Red channel, set the LSB to the corresponding bit of the character

**Before:** Red = 180 → `10110100`
**After:**  Red = 181 → `10110101`  (only the last bit changed)

The visual difference is **1 gray level out of 256** — completely imperceptible.

### What this cell does
- Encodes the message 'HELLO' into the Red channel only
- Appends a null byte (ASCII 0) to mark the end of the message
- Displays original vs encoded side by side

### What to observe
- The images look identical
- We need amplification to see any difference at all


In [ ]:
def encode_single_channel(image, message):
    """Hide message in the LSB of the Red channel (index 2 in BGR)."""
    encoded = image.copy()
    h, w = encoded.shape[:2]

    # Convert message to bits (add null terminator)
    bits = []
    for char in message + chr(0):
        byte = format(ord(char), '08b')
        bits.extend([int(b) for b in byte])

    total_pixels = h * w
    if len(bits) > total_pixels:
        raise ValueError(f"Message too long! Need {len(bits)} pixels, have {total_pixels}.")

    # Write bits into LSB of Red channel
    idx = 0
    for row in range(h):
        for col in range(w):
            if idx >= len(bits):
                break
            pixel = encoded[row, col].copy()
            # Red channel is index 2 in BGR
            pixel[2] = (int(pixel[2]) & 0xFE) | bits[idx]
            encoded[row, col] = pixel
            idx += 1
        if idx >= len(bits):
            break

    return encoded

message = "HELLO"
encoded_single = encode_single_channel(img, message)

show([img, encoded_single], ["Original", f"Encoded: '{message}' (Red LSB)"])
print(f"Message: '{message}' ({len(message)} chars = {len(message)*8 + 8} bits including null terminator)")
print(f"Pixels modified: {len(message)*8 + 8}")
print(f"Max pixel value change: {int(np.max(cv2.absdiff(img, encoded_single)))}")


## Cell 6 — Difference detection (amplification)

### Concept
The raw difference between original and encoded images is at most 1 per modified pixel — invisible on screen.

By **multiplying the difference by 50**, we amplify it to a visible range. A forensic analyst uses this technique to detect steganography.

### What this cell does
- Computes absolute pixel difference between original and encoded
- Multiplies by 50 to amplify
- Displays the amplified difference image

### What to observe
- The pattern reveals exactly which pixels were modified and which channel was touched
- The positions correspond to where the message bits were written (top-left region)


In [ ]:
# Compute and amplify the difference
diff = cv2.absdiff(img, encoded_single)
amplified = np.clip(diff.astype(np.int32) * 50, 0, 255).astype(np.uint8)

show([img, encoded_single, amplified],
     ["Original", "Encoded", "Difference x50"])

print(f"Max difference:  {int(np.max(diff))}")
print(f"Mean difference: {np.mean(diff):.6f}")
print(f"Pixels changed:  {int(np.count_nonzero(diff[:,:,2]))}")  # Red channel
print()
print("The pattern in the amplified image shows exactly where bits were written.")
print("Notice: only the Red channel has changes (the B and G channels are untouched).")


## Cell 7 — Multi-channel encoding

### Concept
Instead of using only the Red channel, we spread bits across all three channels:
- **Pixel 0:** bit 0 → Blue, bit 1 → Green, bit 2 → Red
- **Pixel 1:** bit 3 → Blue, bit 4 → Green, bit 5 → Red
- ...

This stores **3 bits per pixel** instead of 1, tripling the capacity.

### What this cell does
- Encodes 'VISION' using all 3 channels
- Shows the amplified difference — now all 3 channels are modified

### Why it matters
Real steganography tools almost always use multi-channel encoding to maximize capacity.


In [ ]:
def encode_multi_channel(image, message):
    """Hide message across B, G, R channels (3 bits per pixel)."""
    encoded = image.copy()
    h, w = encoded.shape[:2]

    bits = []
    for char in message + chr(0):
        byte = format(ord(char), '08b')
        bits.extend([int(b) for b in byte])

    if len(bits) > h * w * 3:
        raise ValueError("Message too long for multi-channel encoding.")

    idx = 0
    for row in range(h):
        for col in range(w):
            if idx >= len(bits):
                break
            pixel = encoded[row, col].copy()
            for ch in range(3):  # B=0, G=1, R=2
                if idx < len(bits):
                    pixel[ch] = (int(pixel[ch]) & 0xFE) | bits[idx]
                    idx += 1
            encoded[row, col] = pixel
        if idx >= len(bits):
            break

    return encoded

message2 = "VISION"
encoded_multi = encode_multi_channel(img, message2)

diff_multi = cv2.absdiff(img, encoded_multi)
amplified_multi = np.clip(diff_multi.astype(np.int32) * 50, 0, 255).astype(np.uint8)

show([img, encoded_multi, amplified_multi],
     ["Original", f"Encoded: '{message2}' (3-ch)", "Difference x50"])

total_bits = img.shape[0] * img.shape[1] * 3
print(f"Multi-channel capacity: {total_bits:,} bits = {total_bits // 8:,} characters")
print(f"Pixels changed (any channel): {int(np.count_nonzero(np.any(diff_multi > 0, axis=2)))}")


## Cell 8 — Decoding: extract the hidden message

### Concept
Decoding is the reverse of encoding:
1. Read the LSB of each pixel channel in the same order
2. Group bits into 8-bit bytes
3. Convert each byte to its ASCII character
4. Stop when you read the null character (ASCII 0 = `00000000`)

### What this cell does
- Decodes the message hidden in `encoded_single` (Red channel only)
- Then decodes the multi-channel encoded image
- Prints the recovered messages

### Why it matters
If encoding and decoding are correct, the original message is recovered exactly.


In [ ]:
def decode_single_channel(image):
    """Read message from LSB of Red channel."""
    h, w = image.shape[:2]
    bits = []
    for row in range(h):
        for col in range(w):
            bits.append(int(image[row, col, 2]) & 1)  # Red = index 2

    message = ""
    for i in range(0, len(bits) - 7, 8):
        byte = int(''.join(str(b) for b in bits[i:i+8]), 2)
        if byte == 0:
            break
        message += chr(byte)
    return message

def decode_multi_channel(image):
    """Read message from LSB of B, G, R channels."""
    h, w = image.shape[:2]
    bits = []
    for row in range(h):
        for col in range(w):
            for ch in range(3):
                bits.append(int(image[row, col, ch]) & 1)

    message = ""
    for i in range(0, len(bits) - 7, 8):
        byte = int(''.join(str(b) for b in bits[i:i+8]), 2)
        if byte == 0:
            break
        message += chr(byte)
    return message

recovered_single = decode_single_channel(encoded_single)
recovered_multi = decode_multi_channel(encoded_multi)

print(f"Single-channel encoded message:  '{message}'")
print(f"Single-channel decoded message:  '{recovered_single}'")
print(f"Match: {message == recovered_single}")
print()
print(f"Multi-channel encoded message:  '{message2}'")
print(f"Multi-channel decoded message:  '{recovered_multi}'")
print(f"Match: {message2 == recovered_multi}")


## Cell 9 — The forensic challenge

### Concept
In real steganography scenarios, you receive an image and don't know:
- Whether it contains hidden data
- What the message is

The first step is **detection** — does the image look "statistically unusual" in its LSBs?

A clean image has LSBs that follow the noise distribution of natural images. A steganographically modified image will often have LSBs that look more uniformly random (because you're overwriting them with message bits).

### What this cell does
- Generates a pre-encoded image containing a secret message
- Visualises the LSB plane of the Red channel as a binary image
- A uniformly grey/noisy LSB plane = likely encoded
- An LSB plane with visible structure = likely natural (not encoded)

### What to observe
- Compare the LSB plane of the original image vs the encoded image
- Where the message was written, the LSB plane looks like random noise


In [ ]:
# ── do not modify this cell ──
# Pre-encode a secret message into a new image for the forensic challenge
secret_msg = "STEGO"
challenge_image = encode_single_channel(img.copy(), secret_msg)

# Extract and display LSB planes (Red channel)
def lsb_plane(image):
    """Extract the LSB of the Red channel as a binary image (0 or 255)."""
    lsb = (image[:, :, 2] & 1) * 255
    return lsb.astype(np.uint8)

lsb_original = lsb_plane(img)
lsb_encoded  = lsb_plane(challenge_image)

show([img, lsb_original, challenge_image, lsb_encoded],
     ["Original", "LSB plane — original", "Encoded", "LSB plane — encoded"], cols=4)

print("Left: original image LSB plane — follows natural image statistics.")
print("Right: encoded image LSB plane — the top-left region looks uniformly random.")
print("That region is where the message bits were written.")


## Summary

### What you learned

| Concept | Key point |
|---|---|
| LSB steganography | Modifying the last bit changes pixel value by at most 1 — invisible to the eye |
| Capacity | 1 bit/channel/pixel → H×W characters per channel |
| Multi-channel | Use B, G, R to triple capacity |
| Detection | Amplify the difference or inspect the LSB plane |
| Decoding | Read bits in the same order they were written, group into bytes |

### Main question

**Which change is harder to detect — single-channel or multi-channel encoding?**
Both change pixel values by at most 1, but multi-channel encoding modifies more channels per pixel.
The amplified difference shows that multi-channel encoding leaves a more visible pattern across all channels, but neither is detectable without the original image.

---
*In the independent work notebook, you will apply these techniques on your own.*
